### Importing necessary modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
warnings.filterwarnings('ignore')
%matplotlib inline
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

### Avoiding restrictions on displaying number of rows and columns

In [ ]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None

### Loading train and test csv files in two separate dataframes

In [ ]:
df_train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
df_test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

### Plotting histogram of target variable(SalePrice) to observe whether it is normally distributed or not

In [ ]:
sns.distplot(df_train['SalePrice'])

### Finding skewness and kurtosis value for target column

In [ ]:
print('Skewness: {}'.format(df_train.SalePrice.skew()))
print('Kurtosis: {}'.format(df_train.SalePrice.kurt()))

### Finding correlation matrix of train dataframe

In [ ]:
corrmat = df_train.corr()

### Plotting heatmap for top ten correlated attributes(variables) of train dataset

In [ ]:
k = 12
cols = corrmat.nlargest(k, 'SalePrice') ['SalePrice'].index
cm = np.corrcoef(df_train[cols].values.T)
fig, ax = plt.subplots(figsize=(10,10))
sns.set(font_scale = 1)
hm = sns.heatmap(cm, cbar = True, annot = True, square = True, fmt = '.2f', annot_kws = {'size': 12}, 
                 yticklabels = cols.values, xticklabels = cols.values)
plt.show()

### Finding relevant features (features which are highly related to target variable) either directly or inversely affecting the target variable (SalePrice)

In [ ]:
cor_target = abs(corrmat["SalePrice"])
relevant_features = cor_target[cor_target>0.3]
relevant_features
# corrmat

### Function to check correlation value between different variables

In [ ]:
# def check_corr(s):
#     print(df_train[['YearBuilt', s]].corr())
    
# for i in cols_train:
#     check_corr(i)

In [ ]:
# print(df_train[['OverallQual', 'LotFrontage']].corr())
# print(df_train[['OverallQual', 'YearBuilt']].corr())
# print(df_train[['OverallQual','YearRemodAdd']].corr())
# print(df_train[['OverallQual','MasVnrArea']].corr())
# print(df_train[['OverallQual','BsmtFinSF1']].corr())
# print(df_train[['OverallQual','TotalBsmtSF']].corr())
# print(df_train[['OverallQual','1stFlrSF']].corr())
# print(df_train[['OverallQual','2ndFlrSF']].corr())
# print(df_train[['OverallQual','GrLivArea']].corr())
# print(df_train[['OverallQual','FullBath']].corr())
# print(df_train[['OverallQual','TotRmsAbvGrd']].corr())
# print(df_train[['OverallQual','Fireplaces']].corr())
# print(df_train[['OverallQual','GarageYrBlt']].corr())
# print(df_train[['OverallQual','GarageCars']].corr())
# print(df_train[['OverallQual','GarageArea']].corr())
# print(df_train[['OverallQual','WoodDeckSF']].corr())
# print(df_train[['OverallQual','OpenPorchSF']].corr())

### Storing most correlated attributes with respect to target column (both in train and test dataset) in cols_train and cols_test variables

In [ ]:
cols_train = ['SalePrice', 'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',
                          '1stFlrSF', 'FullBath','TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt',
                          'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'LotFrontage', 'WoodDeckSF', '2ndFlrSF',
                          'OpenPorchSF']
cols_test = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',
                          '1stFlrSF', 'FullBath','TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt',
                          'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'LotFrontage', 'WoodDeckSF', '2ndFlrSF',
                          'OpenPorchSF']

### With respect to above filtered attributes, making the final dataframe for both test and train dataset

In [ ]:
df_train_final = df_train[cols_train]
df_test_final = df_test[cols_test]

### Plotting the above selected attributes with respect to every other selected attributes

In [ ]:
sns.set()
sns.pairplot(df_train_final[cols_train], size=2.5)
plt.show()

### Finding and removing missing values from train dataset

In [ ]:
total = df_train_final.isnull().sum().sort_values(ascending = False)
percent = (df_train_final.isnull().sum() / df_train_final.isnull().count()).sort_values(ascending = False)
missing_data = pd.concat([total, percent], axis=1, keys = ['Total', 'Percent'])
# missing_data

### Finding and removing missing values from test dataset

In [ ]:
total = df_test_final.isnull().sum().sort_values(ascending = False)
percent = (df_test_final.isnull().sum() / df_test_final.isnull().count()).sort_values(ascending = False)
missing_data = pd.concat([total, percent], axis=1, keys = ['Total', 'Percent'])

In [ ]:
df_train_final = df_train_final.drop((missing_data[missing_data['Total'] > 7]).index,1)
df_train_final.isnull().sum().max()

In [ ]:
df_test_final = df_test_final.drop((missing_data[missing_data['Total'] > 14]).index,1)
df_test_final = df_test_final.fillna(0)
df_test_final.isnull().sum().max()

### Finding the range of variation in target column (SalePrice)

In [ ]:
sc = StandardScaler()
# sc1 = StandardScaler()
saleprice_scaled = sc.fit_transform(df_train_final['SalePrice'][:,np.newaxis])
low_range = saleprice_scaled[saleprice_scaled[:,0].argsort()][:10]
high_range = saleprice_scaled[saleprice_scaled[:,0].argsort()][-10:]

### Bivariate analysis

### Scatter plot for numerical variables

In [ ]:
var = 'GrLivArea'
data = pd.concat([df_train_final['SalePrice'], df_train_final[var]], axis = 1)
data.plot.scatter(x = var, y = 'SalePrice', ylim = (0, 800000))

### Values corresponding to following two Id's are outliers, so can be removed

In [ ]:
# df_train_final.sort_values(by = 'GrLivArea', ascending=False)[:2]
# df_train_final = df_train_final.drop([1298, 523])
# df_test_final.sort_values(by = 'GrLivArea', ascending=False)[:2]
# df_test_final = df_test_final.drop([1089, 728])

In [ ]:
var = 'TotalBsmtSF'
data = pd.concat([df_train_final['SalePrice'], df_train_final[var]], axis = 1)
data.plot.scatter(x = var, y = 'SalePrice', ylim = (0, 800000))

### Boxplot for categorical variables

In [ ]:
var = 'OverallQual'
data = pd.concat([df_train_final['SalePrice'], df_train_final[var]], axis = 1)
f, ax = plt.subplots(figsize = (8, 5))
fig = sns.boxplot(x = var, y = 'SalePrice', data = data)
fig.axis(ymin = 0, ymax = 800000)

In [ ]:
var = 'YearBuilt'
data = pd.concat([df_train_final['SalePrice'], df_train_final[var]], axis = 1)
f, ax = plt.subplots(figsize = (8, 5))
fig = sns.boxplot(x = var, y = 'SalePrice', data = data)
fig.axis(ymin = 0, ymax = 800000)

In [ ]:
sns.distplot(df_train_final['SalePrice'], fit = norm)
fig = plt.figure()
res = stats.probplot(df_train_final['SalePrice'], plot = plt)

### Plotting histogram for GrLivArea column, to find whether it is normally distributed or skewed

In [ ]:
sns.distplot(df_train_final['GrLivArea'], fit = norm)
fig = plt.figure()
res = stats.probplot(df_train_final['GrLivArea'], plot = plt)

### As skewness is there, which we can observe from above plots, so doing log transformation will help to avoid this

In [ ]:
df_train_final['GrLivArea'] = np.log(df_train_final['GrLivArea'])
df_test_final['GrLivArea'] = np.log(df_test_final['GrLivArea'])

In [ ]:
sns.distplot(df_train_final['GrLivArea'], fit = norm)
fig = plt.figure()
res = stats.probplot(df_train_final['GrLivArea'], plot = plt)

### Plotting histogram for TotalBsmtSF column, to observe whether skewness is there or it is normally distributed

In [ ]:
sns.distplot(df_train_final['TotalBsmtSF'], fit = norm)
fig = plt.figure()
res = stats.probplot(df_train_final['TotalBsmtSF'], plot = plt)

### As zero values are present in this column, so log transformation won't work in this case, so first we are creating one binary variable and then doing log transformation to remove skewness

In [ ]:
df_train_final['HasBsmt'] = pd.Series(len(df_train_final['TotalBsmtSF']), index = df_train.index)
df_test_final['HasBsmt'] = pd.Series(len(df_test_final['TotalBsmtSF']), index = df_test.index)
df_train_final['HasBsmt'] = 0
df_test_final['HasBsmt'] = 0
df_train_final.loc[df_train_final['TotalBsmtSF']>0,'HasBsmt'] = 1
df_test_final.loc[df_test_final['TotalBsmtSF']>0,'HasBsmt'] = 1

In [ ]:
df_train_final.loc[df_train_final['HasBsmt'] == 1, 'TotalBsmtSF'] = np.log(df_train_final['TotalBsmtSF'])
df_test_final.loc[df_test_final['HasBsmt'] == 1, 'TotalBsmtSF'] = np.log(df_test_final['TotalBsmtSF'])

In [ ]:
sns.distplot(df_train_final[df_train_final['TotalBsmtSF'] > 0]['TotalBsmtSF'], fit = norm)
fig = plt.figure()
res = stats.probplot(df_train_final[df_train['TotalBsmtSF'] > 0]['TotalBsmtSF'], plot = plt)

### Plotting GrLivArea column with respect to SalePrice (target column) to find the relationship between them

In [ ]:
plt.scatter(df_train_final['GrLivArea'], df_train_final['SalePrice'])

### Plotting TotalBsmtSF column with respect to SalePrice (target column) to find the relationship between them

In [ ]:
plt.scatter(df_train_final[df_train_final['TotalBsmtSF'] > 0]['TotalBsmtSF'], df_train_final[df_train_final['TotalBsmtSF'] > 0]['SalePrice'])

### Final selection of train and test variables (after removing missing values)

In [ ]:
cols_train_final = ['SalePrice', 'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',
                          '1stFlrSF', 'FullBath','TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd'
                          , 'Fireplaces', 'BsmtFinSF1', 'WoodDeckSF', '2ndFlrSF',
                          'OpenPorchSF']
cols_test_final = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF',
                          '1stFlrSF', 'FullBath','TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd'
                          , 'Fireplaces', 'BsmtFinSF1', 'WoodDeckSF', '2ndFlrSF',
                          'OpenPorchSF']

In [ ]:
df_train_final[cols_train_final].head()

### Creating the final dataframe having final selected columns

In [ ]:
df_train_final = df_train_final[cols_train_final]
df_test_final = df_test_final[cols_test_final]

In [ ]:
df_train_final = df_train_final[cols_train_final]
df_test_final = df_test_final[cols_test_final]

### If you want to do scaling, before passing to model

In [ ]:
# sc.fit(df_train_final)
# sc1.fit(df_test_final)
# tranformed_df = pd.DataFrame(df_train_final_scaled_X, columns=cols_train_final)
# tranformed_df_test = pd.DataFrame(df_test_final_scaled_X, columns=cols_train_final)
# final_train_data = tranformed_df.copy()
# final_test_data = tranformed_df_test.copy()
# final_train_data[['GrLivArea', 'TotalBsmtSF', 'SalePrice']] = df_train_final[['GrLivArea', 'TotalBsmtSF', 'SalePrice']]
# final_test_data[['GrLivArea', 'TotalBsmtSF']] = df_test_final[['GrLivArea', 'TotalBsmtSF']]
# final_train_data.head()

### Creating dummies for categorical variables

In [ ]:
final_train_data_dummies = pd.get_dummies(df_train_final)
final_test_data_dummies = pd.get_dummies(df_test_final)

### Random Forest Regressor model

In [ ]:
rf = RandomForestRegressor(n_estimators=20)

### Creating dependent and independent variables for input in model

In [ ]:
X = final_train_data_dummies.loc[:,final_train_data_dummies.columns != 'SalePrice']
y = final_train_data_dummies.SalePrice

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### Fitting the created model

In [ ]:
rf_fit = rf.fit(X,y)

### Making prediction on test data

In [ ]:
y = rf_fit.predict(final_test_data_dummies)

### Creating dataframe for predicted values

In [ ]:
submission_df = pd.DataFrame(y,columns=['SalePrice'])

### Final dataframe containing Id and corresponding predicted SalePrice columns for test dataset

In [ ]:
submission_df['Id'] = df_test['Id']
submission_df = submission_df[['Id', 'SalePrice']]

### Saving final dataset to specified folder in csv format

In [ ]:
submission_df.to_csv('/kaggle/working/submission.csv', index=False)